# Practice 4 (Week 4): Extend the three patterns
# تمرین تمرینی ۴ (هفته‌ی ۴): گسترش سه الگوی طراحی

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0 &nbsp;|&nbsp;
**Released:** Week 4 &nbsp;|&nbsp; **Due:** start of Week 5 &nbsp;|&nbsp; **Effort:** about 3 hours &nbsp;|&nbsp;
**Weight:** formative (feedback only; the graded milestones are A1 in Week 2 and A2 in Week 5)

> **Provenance:** new content. Patterns rewritten from GeeksforGeeks ("Strategy Design Pattern",
> "Observer Design Pattern", "Factory Method - Python Design Patterns", "Pytest Fixtures") and
> Gamma et al., *Design Patterns* (1994); polynomial features adapted from Bishop, *PRML* (2006),
> Sec. 1.1 and 3.1; fuel-flow model adapted from Anderson, *Aircraft Performance and Design*, Ch. 5.

<div dir="rtl">

## صورت مسئله

در کلاس دیدیم که مدل خطی، جریان سوخت پهپاد را با خطای RMSE حدود ۵۱ گرم بر ساعت پیش‌بینی می‌کند، در حالی که نویز اندازه‌گیری فقط حدود ۱۲ گرم بر ساعت است. دلیل فیزیکی روشن است: توان پسای مزاحم با مکعب سرعت رشد می‌کند و توان صعود حاصل‌ضرب جرم در نرخ صعود است؛ هیچ‌کدام خطی نیستند. در این تمرین هر سه الگوی هفته را **بدون ویرایش حتی یک خط از کتابخانه‌ی درس** گسترش می‌دهید:

۱. **راهبرد (Strategy):** کلاس <code>PolynomialLeastSquares</code> را بنویسید که ویژگی‌ها را استاندارد کرده، توان‌های آن‌ها و (به‌دلخواه) حاصل‌ضرب‌های دوتایی را می‌سازد و حل خطی را به یک شیء <code>LinearLeastSquares</code> واگذار می‌کند (ترکیب به‌جای وراثت).

۲. **ناظر (Observer):** کالبک <code>ReduceLROnPlateau</code> را بنویسید که اگر خطای اعتبارسنجی چند دوره بهبود نیابد، نرخ یادگیری را کاهش دهد.

۳. **کارخانه (Factory):** بارگذار <code>FlightRegimeLoader</code> را با نام <code>"fuel_flow_regime"</code> در کارخانه ثبت کنید؛ این بارگذار هر مجموعه‌داده‌ی ثبت‌شده‌ی دیگری را از طریق خود کارخانه می‌سازد و فقط سطرهای یک رژیم پروازی (کروز، صعود یا فرود) را نگه می‌دارد.

۴. **آزمون (pytest):** دست‌کم شش آزمون دیگر در <code>test_patterns_practice.py</code> بنویسید.

کد را در فایل <code>patterns_practice.py</code> بنویسید؛ این دفترچه فقط ماژول شما را بارگذاری و بررسی می‌کند.

</div>

**In English.** The linear model predicts fuel flow with a test RMSE of about 51 g/h although the
measurement noise is only about 12 g/h: parasite power grows with $V^3$ and climb power is
mass × climb rate. Extend each pattern **without editing the course library**: a polynomial
strategy (Task 1), a plateau learning-rate callback (Task 2), a regime-filtering loader registered
in the factory (Task 3), and your own tests (Task 4). Code goes in `patterns_practice.py`.

## Setup (run first, do not edit)

In [ ]:
from __future__ import annotations

import importlib
import subprocess
import sys
from pathlib import Path
from types import ModuleType
from typing import Final

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
ASSIGNMENT_DIR: Final[Path] = REPO_ROOT / "assignments" / "practice04"
for folder in (REPO_ROOT / "src", ASSIGNMENT_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from aerospace_ai.datasets import DatasetFactory, UnknownDatasetError  # noqa: E402
from aerospace_ai.models import LinearLeastSquares, RegressionExperiment  # noqa: E402
from aerospace_ai.training import History, Trainer  # noqa: E402
from aerospace_ai.models import GradientDescentRegressor  # noqa: E402

SEED: Final[int] = 4
fuel = DatasetFactory.load("fuel_flow_synthetic", seed=SEED)
train, test = fuel.split()


def load_student_module() -> ModuleType:
    """(Re)import patterns_practice.py so edits are picked up without a kernel restart."""
    if "fuel_flow_regime" in DatasetFactory.available():
        DatasetFactory.unregister("fuel_flow_regime")   # the module registers it again
    if "patterns_practice" in sys.modules:
        return importlib.reload(sys.modules["patterns_practice"])
    return importlib.import_module("patterns_practice")


def status(task: str, check: object) -> None:
    """Run a public check and print a short status line."""
    try:
        check()
        print(f"[x] {task}: public checks passed")
    except NotImplementedError:
        print(f"[ ] {task}: not done yet")
    except UnknownDatasetError:
        print(f"[ ] {task}: not done yet (loader not registered)")
    except AssertionError as err:
        print(f"[!] {task}: check failed: {err}")


print("Repository root found:", REPO_ROOT.name, "|", fuel)

### Task 1 (30 pts): `PolynomialLeastSquares`
<div dir="rtl">ویژگی‌ها را با میانگین و انحراف معیار داده‌ی آموزش استاندارد کنید، بلوک‌های توان ۱ تا degree و سپس حاصل‌ضرب‌های دوتایی را بسازید و برازش را به <code>self.linear</code> بسپارید.</div>

By hand first: with two features and `degree=2, interactions=True` the expansion has
$2 + 2 + 1 = 5$ columns; for $X = [[1, 10], [3, 30]]$ both columns standardise to $z = (-1, +1)$,
so the rows are $(-1, -1, 1, 1, 1)$ and $(1, 1, 1, 1, 1)$.

In [ ]:
def _check_task1() -> None:
    pp = load_student_module()
    model = pp.PolynomialLeastSquares(2, True).fit(np.array([[1.0, 10.0], [3.0, 30.0]]), np.zeros(2))
    print(model, "\n", model.expand(np.array([[1.0, 10.0], [3.0, 30.0]])))
    rows = []
    for strategy in (LinearLeastSquares(), pp.PolynomialLeastSquares(2, False), pp.PolynomialLeastSquares(2, True)):
        rows.append(RegressionExperiment(strategy).run(train, test))
    display(pd.DataFrame(rows).set_index("model").round(2))
    assert rows[2].test_rmse < rows[0].test_rmse, "the polynomial model should beat the line"


status("Task 1", _check_task1)

In [ ]:
# HIDDEN TESTS (task1): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 2 (30 pts): `ReduceLROnPlateau`
<div dir="rtl">اگر خطای اعتبارسنجی به اندازه‌ی patience دوره بهتر از best − min_delta نشود، نرخ یادگیری را در factor ضرب کنید (نه کمتر از min_lr)، شماره‌ی دوره را در <code>reductions</code> ثبت و شمارنده را صفر کنید.</div>

By hand first: losses $5, 4, 4, 4, 3, 3.5, 3.4, 3.2, 3.1$ with `patience=2, factor=0.5,
min_lr=0.2` and start rate 1.0 give reductions after epochs 3, 6 and 8 (rates 0.5, 0.25, 0.2).

In [ ]:
def _check_task2() -> None:
    pp = load_student_module()
    fit, val = train.split()
    plateau, history = pp.ReduceLROnPlateau(patience=3, factor=0.5, min_delta=1.0), History()
    Trainer([plateau, history]).fit(GradientDescentRegressor(0.1), fit.X, fit.y, val.X, val.y, max_epochs=100)
    print("reductions after epochs:", plateau.reductions)
    print("final learning rate:", history.learning_rate[-1])
    assert plateau.reductions, "no reduction happened"


status("Task 2", _check_task2)

In [ ]:
# HIDDEN TESTS (task2): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 3 (25 pts): `FlightRegimeLoader` in the factory
<div dir="rtl">بارگذار را با دکوراتور <code>@DatasetFactory.register("fuel_flow_regime")</code> ثبت کنید. مجموعه‌داده‌ی پایه را با <code>DatasetFactory.load(base, **base_kwargs)</code> بسازید و سطرهای رژیم را با <code>subset</code> جدا کنید.</div>

In [ ]:
def _check_task3() -> None:
    pp = load_student_module()
    sizes = {r: len(DatasetFactory.load("fuel_flow_regime", regime=r, seed=SEED)) for r in pp.REGIMES}
    print(sizes)
    assert sum(sizes.values()) == len(fuel), "the three regimes must partition the data"


status("Task 3", _check_task3)

In [ ]:
# HIDDEN TESTS (task3): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 4 (15 pts): your pytest suite
<div dir="rtl">در <code>test_patterns_practice.py</code> دست‌کم شش آزمون دیگر بنویسید و از fixture، <code>parametrize</code> و <code>pytest.raises</code> استفاده کنید.</div>

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "test_patterns_practice.py"],
    cwd=ASSIGNMENT_DIR, capture_output=True, text=True,
)
print(result.stdout[-2000:])

In [ ]:
# HIDDEN TESTS (task4): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

## Expected output (check yourself against these)

| Quantity | Expected value |
|----------|----------------|
| `expand([[1, 10], [3, 30]])`, degree 2, interactions | `[[-1, -1, 1, 1, 1], [1, 1, 1, 1, 1]]` |
| Test RMSE: linear / poly-2 / poly-2 + interactions | 51.46 / 41.02 / 22.59 g/h |
| Plateau run (patience 3, factor 0.5, min_delta 1) | first reduction after epoch 32, 17 reductions, final rate 1e-06 |
| Hand trace of Task 2 | reductions after epochs [3, 6, 8] |
| Regime sizes (seed 4) | cruise 80, climb 199, descent 121 |
| Your pytest suite | at least 8 tests, all passing |

**Reflection (not graded, 3 sentences in a Markdown cell).** Which product feature explains most
of the gain in Task 1, and why does physics predict it? Hint: climb power.